# Notebook 4: Group Comparisons

## Purpose
Test hypotheses H1–H5 across popularity groups.

## Tests
- **Kruskal-Wallis** for non-parametric group comparison
- **Post-hoc tests** with Holm correction
- **Effect sizes**: Eta-squared, Cohen's d
- **Group-wise violin plots** of category proportions and indices

## Aligned Hypotheses
- H1: `(Love-over-Sex index) → Top > Trash`
- H2: `(HEA Index) → Top > Trash`
- H4: `(Protective–Jealousy Delta) → Top > Trash`
- H5: `(Darkness–Tenderness) → Top < Trash`

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

np.random.seed(42)

## 1. Load Data

In [ ]:
PROJECT_ROOT = Path().resolve().parent.parent.parent.parent
INPUT_FILE = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis" / "indices_book.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} books")
if 'group' in df.columns:
    print(f"Groups: {df['group'].value_counts().to_dict()}")

## 2. Define Indices to Test

In [ ]:
# Define indices and their expected direction for Top vs Trash
hypotheses = {
    'love_over_sex': {'hypothesis': 'H1', 'direction': 'Top > Trash'},
    'hea_index': {'hypothesis': 'H2', 'direction': 'Top > Trash'},
    'protective_minus_jealous': {'hypothesis': 'H4', 'direction': 'Top > Trash'},
    'dark_vs_tender': {'hypothesis': 'H5', 'direction': 'Top < Trash'},
}

# Filter to available indices
available_hypotheses = {k: v for k, v in hypotheses.items() if k in df.columns}
print(f"Testing {len(available_hypotheses)} hypotheses")

## 3. Kruskal-Wallis Tests

In [ ]:
def kruskal_wallis_with_effect_size(data, groups, variable):
    """Perform Kruskal-Wallis test and compute eta-squared effect size."""
    group_data = [data[data[groups] == g][variable].dropna() for g in data[groups].unique()]
    
    # Kruskal-Wallis test
    h_stat, p_value = kruskal(*group_data)
    
    # Eta-squared (approximate)
    n_total = len(data[variable].dropna())
    eta_squared = (h_stat - len(group_data) + 1) / (n_total - len(group_data))
    eta_squared = max(0, eta_squared)  # Ensure non-negative
    
    return {
        'variable': variable,
        'h_statistic': h_stat,
        'p_value': p_value,
        'eta_squared': eta_squared,
        'n_groups': len(group_data),
        'n_total': n_total
    }

# Run tests
results = []
if 'group' in df.columns:
    for idx, info in available_hypotheses.items():
        result = kruskal_wallis_with_effect_size(df, 'group', idx)
        result['hypothesis'] = info['hypothesis']
        result['expected_direction'] = info['direction']
        results.append(result)

kruskal_results = pd.DataFrame(results)
print("Kruskal-Wallis Results:")
print(kruskal_results)

## 4. Post-hoc Tests with Holm Correction

In [ ]:
def pairwise_comparisons(data, groups, variable, correction='holm'):
    """Perform pairwise Mann-Whitney U tests with multiple comparison correction."""
    group_names = sorted(data[groups].unique())
    comparisons = []
    
    for i, g1 in enumerate(group_names):
        for g2 in group_names[i+1:]:
            data1 = data[data[groups] == g1][variable].dropna()
            data2 = data[data[groups] == g2][variable].dropna()
            
            if len(data1) > 0 and len(data2) > 0:
                u_stat, p_val = mannwhitneyu(data1, data2, alternative='two-sided')
                
                # Cohen's d
                pooled_std = np.sqrt(((len(data1) - 1) * data1.std()**2 + 
                                    (len(data2) - 1) * data2.std()**2) / 
                                   (len(data1) + len(data2) - 2))
                cohens_d = (data1.mean() - data2.mean()) / pooled_std if pooled_std > 0 else 0
                
                comparisons.append({
                    'variable': variable,
                    'group1': g1,
                    'group2': g2,
                    'u_statistic': u_stat,
                    'p_value': p_val,
                    'cohens_d': cohens_d,
                    'mean_diff': data1.mean() - data2.mean()
                })
    
    # Apply multiple comparison correction
    if comparisons:
        p_values = [c['p_value'] for c in comparisons]
        _, p_corrected, _, _ = multipletests(p_values, method=correction)
        for i, comp in enumerate(comparisons):
            comp['p_value_corrected'] = p_corrected[i]
    
    return pd.DataFrame(comparisons)

# Run pairwise comparisons
if 'group' in df.columns:
    posthoc_results = []
    for idx in available_hypotheses.keys():
        pairwise = pairwise_comparisons(df, 'group', idx)
        posthoc_results.append(pairwise)
    
    posthoc_df = pd.concat(posthoc_results, ignore_index=True)
    print("\nPost-hoc Pairwise Comparisons:")
    print(posthoc_df)

## 5. Violin Plots

In [ ]:
# Create violin plots for each index
if 'group' in df.columns:
    for idx in available_hypotheses.keys():
        plt.figure(figsize=(10, 6))
        sns.violinplot(data=df, x='group', y=idx, order=sorted(df['group'].unique()))
        plt.title(f'{idx} by Group\n{available_hypotheses[idx]["hypothesis"]}: {available_hypotheses[idx]["direction"]}')
        plt.ylabel('Index Value')
        plt.xlabel('Group')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'{idx}_violin_by_group.png')
        plt.show()

## 6. Volcano Plot

In [ ]:
# Volcano plot: effect size vs -log10(p-value)
if 'group' in df.columns and 'Top' in df['group'].values and 'Trash' in df['group'].values:
    volcano_data = []
    for idx in available_hypotheses.keys():
        top_data = df[df['group'] == 'Top'][idx].dropna()
        trash_data = df[df['group'] == 'Trash'][idx].dropna()
        
        if len(top_data) > 0 and len(trash_data) > 0:
            u_stat, p_val = mannwhitneyu(top_data, trash_data, alternative='two-sided')
            effect_size = (top_data.mean() - trash_data.mean()) / np.sqrt(
                ((len(top_data) - 1) * top_data.std()**2 + (len(trash_data) - 1) * trash_data.std()**2) /
                 (len(top_data) + len(trash_data) - 2)
            )
            
            volcano_data.append({
                'index': idx,
                'effect_size': effect_size,
                'neg_log10_p': -np.log10(max(p_val, 1e-10)),
                'p_value': p_val
            })
    
    if volcano_data:
        volcano_df = pd.DataFrame(volcano_data)
        
        plt.figure(figsize=(10, 8))
        plt.scatter(volcano_df['effect_size'], volcano_df['neg_log10_p'], s=100, alpha=0.7)
        for _, row in volcano_df.iterrows():
            plt.annotate(row['index'], (row['effect_size'], row['neg_log10_p']))
        plt.axhline(-np.log10(0.05), color='r', linestyle='--', label='p=0.05')
        plt.xlabel('Effect Size (Cohen\'s d)')
        plt.ylabel('-log10(p-value)')
        plt.title('Volcano Plot: Effect Sizes vs Significance')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'volcano_plot.png')
        plt.show()

## 7. Save Results

In [ ]:
# Save Kruskal-Wallis results
if 'kruskal_results' in locals():
    output_file = OUTPUT_DIR / "kruskal_wallis_results.csv"
    kruskal_results.to_csv(output_file, index=False)
    print(f"✓ Saved: {output_file}")

# Save post-hoc results
if 'posthoc_df' in locals():
    output_file = OUTPUT_DIR / "posthoc_pairwise_results.csv"
    posthoc_df.to_csv(output_file, index=False)
    print(f"✓ Saved: {output_file}")

## Summary

Group comparisons complete. Next: Notebook 5 (Modeling & Prediction)